# Experiment 5.4.3 — Causal elapsed-time readout capacity

**Analysis-only notebook.** Training, selection, and final-test execution live in the Python/Slurm pipeline. This notebook only reads finalized CSV/JSON artifacts.

Question: can a compact causal elapsed-time-conditioned readout recover the phase-specific weighting of frozen-WHAT Fixed250 + full Linear?


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('notebooks/artifacts/experiment_5_4_3_elapsed_readout_capacity/elapsed_readout_capacity_v1')
if not ROOT.exists():
    ROOT = Path('artifacts/experiment_5_4_3_elapsed_readout_capacity/elapsed_readout_capacity_v1')
ROOT


In [ ]:
required = ['reference_runs.csv', 'capacity_runs.csv', 'capacity_summary.csv', 'capacity_recovery.csv', 'capacity_selection.json']
missing = [name for name in required if not (ROOT / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing finalized Stage-A artifacts: {missing}')
reference = pd.read_csv(ROOT / 'reference_runs.csv')
capacity = pd.read_csv(ROOT / 'capacity_runs.csv')
summary = pd.read_csv(ROOT / 'capacity_summary.csv')
recovery = pd.read_csv(ROOT / 'capacity_recovery.csv')
capacity_selection = json.loads((ROOT / 'capacity_selection.json').read_text())
display(reference)
display(summary.sort_values('mean_val_balanced_accuracy', ascending=False))
capacity_selection


In [ ]:
pivot = summary[summary.parameterization == 'factorized'].pivot(index='rank', columns='n_banks', values='mean_val_balanced_accuracy')
fig, ax = plt.subplots(figsize=(7, 4.5))
image = ax.imshow(pivot.values, aspect='auto')
ax.set_xticks(range(len(pivot.columns)), [str(v) for v in pivot.columns])
ax.set_yticks(range(len(pivot.index)), [str(v) for v in pivot.index])
ax.set_xlabel('Number of banks K')
ax.set_ylabel('Rank r')
ax.set_title('Validation BA — elapsed-conditioned capacity map')
fig.colorbar(image, ax=ax, label='Balanced accuracy')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ordered = summary.sort_values('trainable_parameter_count')
ax.scatter(ordered.trainable_parameter_count, ordered.mean_oracle_gap_recovery_fraction)
for _, row in ordered.iterrows():
    ax.annotate(f"K{int(row.n_banks)}/r{int(row['rank'])}", (row.trainable_parameter_count, row.mean_oracle_gap_recovery_fraction))
ax.axhline(0.90, linestyle='--')
ax.set_xlabel('Trainable parameters')
ax.set_ylabel('Mean Fixed250 oracle-gap recovery fraction')
ax.set_title('Capacity efficiency')
plt.show()


In [ ]:
if (ROOT / 'final_runs.csv').exists():
    final = pd.read_csv(ROOT / 'final_runs.csv')
    display(final)
    cols = ['base_test_balanced_accuracy', 'fixed250_test_balanced_accuracy', 'anchor_k4_r4_test_balanced_accuracy', 'selected_test_balanced_accuracy', 'oracle_gap_recovery_fraction']
    print(final[cols].mean())
else:
    print('Final test remains unopened / not yet finalized.')
